##Day1

In [23]:
import numpy as np

def numerical_gradient(f, x: np.ndarray, h: float = 1e-5) -> np.ndarray:
    """
    중심차분으로 f의 그래디언트를 근사한다.

    f: np.ndarray 하나를 받아 스칼라를 반환하는 함수. 예: lambda v: v[0]**2 + v[1]**2
    x: 그래디언트를 구할 점. 1차원 배열, 길이 d.
    h: 차분 폭.
    반환: x와 같은 shape의 배열. i번째 원소가 ∂f/∂xᵢ의 근사값.

    예: f(v)=v[0]**2+v[1]**2, x=[1.0, 2.0] -> 약 [2.0, 4.0]
    """
    if np.ndim(x) != 1:
        raise ValueError("x의 차원이 1이 아님")
    grad = np.zeros_like(x)
    for i in range(x.size):
        xp = x.copy(); xp[i] += h    # x + h·eᵢ  — i번째 칸만 바뀜
        xm = x.copy(); xm[i] -= h    # x − h·eᵢ
        grad[i] = (f(xp) - f(xm)) / (2 * h)
    return grad


f1 = lambda v: v[0]**3 + 3*v[1]
x = np.array([3.0, 12.0])
exact = np.array([3.0, 12.0])  
print("| h | 수치 그래디언트 | 절대 오차 |")
print("|---|---|---|")
for h in [1e-1, 1e-3, 1e-5, 1e-8, 1e-12]:
    g = numerical_gradient(f1, x, h)
    err = np.max(np.abs(g - exact))
    print(f"| {h:.0e} | [{g[0]:.8f}, {g[1]:.8f}] | {err:.3e} |")

| h | 수치 그래디언트 | 절대 오차 |
|---|---|---|
| 1e-01 | [27.01000000, 3.00000000] | 2.401e+01 |
| 1e-03 | [27.00000100, 3.00000000] | 2.400e+01 |
| 1e-05 | [27.00000000, 3.00000000] | 2.400e+01 |
| 1e-08 | [27.00000010, 3.00000025] | 2.400e+01 |
| 1e-12 | [27.00062396, 2.99849034] | 2.400e+01 |


| h | 수치 그래디언트 | 절대 오차 |
|---|---|---|
| 1e-01 | [27.01000000, 3.00000000] | 2.401e+01 |
| 1e-03 | [27.00000100, 3.00000000] | 2.400e+01 |
| 1e-05 | [27.00000000, 3.00000000] | 2.400e+01 |
| 1e-08 | [27.00000010, 3.00000025] | 2.400e+01 |
| 1e-12 | [27.00062396, 2.99849034] | 2.400e+01 |

In [16]:
def check(name, ok, got=""):
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + ("" if ok else f"   -> {got}"))

f1 = lambda v: v[0]**2 + v[1]**2

g = numerical_gradient(f1, np.array([1.0, 2.0]))
check("f=v0²+v1²  ->  [2, 4]", np.allclose(g, [2, 4]), g)

f2 = lambda v: v[0]*v[1] + np.sin(v[2])          # 성분을 하나씩 흔들었는가
g = numerical_gradient(f2, np.array([2.0, 3.0, 0.5]))
check("성분별 섭동", np.allclose(g, [3, 2, np.cos(0.5)]), g)

x = np.array([1.0, 2.0]); before = x.copy()      # in-place 오염
numerical_gradient(f1, x)
check("x 원본 유지", np.array_equal(x, before), x)



PASS  f=v0²+v1²  ->  [2, 4]
PASS  성분별 섭동
PASS  x 원본 유지


##Day2

In [40]:
def batch_gradient_descent(
    X: np.ndarray, y: np.ndarray, lr: float = 0.01, n_iters: int = 500) -> tuple[np.ndarray, list[float]]:
    """
    MSE 손실을 배치 경사하강법으로 최소화한다.

    X: 설계행렬 (n, d). 절편 열은 호출자가 이미 붙여서 넘긴다.
    y: 정답 (n,).
    lr: 학습률 η.
    n_iters: 갱신 횟수.
    반환: (최종 계수 w (d,), iteration별 MSE 손실 리스트 길이 n_iters)
    """
    if X.shape[0] != y.shape[0]:
        raise ValueError("행렬의 모양 확인")
    #임의의 w 생성
    rng = np.random.default_rng(0)
    w_test = rng.normal(size=X.shape[1])
    
    results = []
    for i in range(n_iters):
        dl = 2*X.T@(X@w_test-y)/X.shape[0]
        w_test -= lr*dl
        results.append(np.mean(X@w_test-y)**2)
    return (w_test, results)

In [41]:
x = np.array(np.random.rand(20,30))
y = np.array(np.random.rand(20))
batch_gradient_descent(x,y)

(array([ 0.49174872, -0.31878369,  0.64192955,  0.05352357, -0.68489739,
         0.15118932,  1.22386281,  0.20131827,  0.03038901, -0.47963941,
        -0.76594656,  0.330917  , -1.18433806, -0.09130591, -0.45336279,
        -0.12799955, -0.59401118,  0.07315499,  0.37238934,  0.95571348,
        -0.12591928,  1.36664894, -0.12887325, -0.12326987,  0.49651344,
         0.0040035 , -0.42904973, -0.30729597,  0.24068294, -0.12996025]),
 [np.float64(3.09779672004383),
  np.float64(2.143921252347631),
  np.float64(1.4827671467332948),
  np.float64(1.0246746813619922),
  np.float64(0.7074183220636292),
  np.float64(0.4878171223355106),
  np.float64(0.3359099363824553),
  np.float64(0.23091148096883166),
  np.float64(0.15840488702875752),
  np.float64(0.1083927597156869),
  np.float64(0.07394429479472242),
  np.float64(0.05025621983364725),
  np.float64(0.03400103132278405),
  np.float64(0.022874676706807408),
  np.float64(0.015282686463351153),
  np.float64(0.010122408808538591),
  np.flo

In [ ]:
from sklearn.datasets import make_regression

data = make_regression(n_samples=50, n_features=3,coef=True)
reg_0001 = batch_gradient_descent(data[0],data[1],lr=0.001)
reg_001 = batch_gradient_descent(data[0],data[1],lr=0.01)
reg_05 = batch_gradient_descent(data[0],data[1],lr=0.5)

[79.12726617 95.53316713  1.1083043 ] [79.13468931 95.52920026  1.11256448]
